In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

2026-03-10 14:40:50.864051: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-10 14:40:50.867251: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/system/software/code-server/4.107.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cU

In [2]:
import pandas as pd
import numpy as np

In [ ]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=3:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

2026-03-10 14:40:56,850 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 12.00 GiB
2026-03-10 14:40:56,852 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 12.00 GiB


2026-03-10 14:41:01,751 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='a1132u07n03.mghpcc.ycrc.yale.edu:58962', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/tornado/web.py", line 3375, in wrapper
    return method(self, *args, **kwargs)
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-t

In [4]:
client.dashboard_link

'http://127.0.0.1:8787/status'

In [5]:
DATA_ROOT="/nfs/roberts/project/pi_skr2/shared/tabula_data"
pkl_path=f"{DATA_ROOT}/shendure/shendure_ortho_20260306/training_data.pkl"
import pickle as pkl
with open(pkl_path,"rb") as f:
    dat=pkl.load(f)

In [7]:
dat.consider_missing(max_memory_gb=300)

scMPRAforge: INFO: consider_missing expanded 781460 observed rows to 652632845 rows by adding 651851385 zero rows. Estimated peak memory: 293.24 GB (cap=300, factor=4.0).


In [10]:
dat.data

,cell_bc,rep_id,cre_id,cell_type,mpra_bc,umis_mpra_bc
npartitions=1,,,,,,
,string,string,string,string,string,"Sparse[int64, 0]"
,...,...,...,...,...,...


2026-03-10 14:44:51,684 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 9.06 GiB -- Worker memory limit: 12.00 GiB
2026-03-10 14:44:51,907 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 9.72 GiB -- Worker memory limit: 12.00 GiB
2026-03-10 14:44:56,463 - distributed.nanny - WARNING - Restarting worker
2026-03-10 14:45:02,233 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 8.55 GiB -- Worker memory limit: 12.00 GiB
2026-03-10 14:45:02,458 - distributed.worker.me

manually calc reference beta...

In [11]:
client.close()
cluster.close()